In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/house-prices-advanced-regression-techniques/test.csv


**Load the dataset**

In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import pandas as pd
import numpy as np

# Ignore in case of warnings
import warnings
warnings.filterwarnings(action="ignore")

In [3]:
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
df_train = train.copy()

# Load df_test data
df_test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

# Load sample_submission
sample_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv')

In [4]:
print("--------------------Null values ratio-----------------------\n")
display(df_train.isnull().mean()*100)


--------------------Null values ratio-----------------------



Id                0.000000
MSSubClass        0.000000
MSZoning          0.000000
LotFrontage      17.739726
LotArea           0.000000
                   ...    
MoSold            0.000000
YrSold            0.000000
SaleType          0.000000
SaleCondition     0.000000
SalePrice         0.000000
Length: 81, dtype: float64

In [5]:
df_train.columns[df_train.isnull().mean()*100 > 20]

Index(['Alley', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature'], dtype='object')

In [6]:
df_train = df_train.drop(columns=['Alley','FireplaceQu','PoolQC','Fence','MiscFeature'],axis=1)

In [7]:
for column in df_train.columns:
    if df_train[column].dtype == np.object:
        df_train[column].fillna(df_train[column].mode()[0], inplace=True)
    elif df_train[column].dtype == np.float64:
        df_train[column].fillna(df_train[column].median(), inplace=True)
    elif df_train[column].dtype == np.int64:
        df_train[column].fillna(df_train[column].median(), inplace=True)
        

In [8]:
print("---------------Checking for the null value again-----------------\n")
display(df_train.isnull().sum())
print("The totall amount of null values:",df_train.isnull().sum().sum())
print("---------------------The final train data set-------------------\n")
display(df_train)

---------------Checking for the null value again-----------------



Id               0
MSSubClass       0
MSZoning         0
LotFrontage      0
LotArea          0
                ..
MoSold           0
YrSold           0
SaleType         0
SaleCondition    0
SalePrice        0
Length: 76, dtype: int64

The totall amount of null values: 0
---------------------The final train data set-------------------



,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,0,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,Reg,Lvl,AllPub,FR2,...,0,0,0,0,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,IR1,Lvl,AllPub,Inside,...,0,0,0,0,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,IR1,Lvl,AllPub,Corner,...,272,0,0,0,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,IR1,Lvl,AllPub,FR2,...,0,0,0,0,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,0,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,0,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,0,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,Reg,Lvl,AllPub,Inside,...,112,0,0,0,0,4,2010,WD,Normal,142125


In [9]:
from sklearn.preprocessing import LabelEncoder
label_encode = LabelEncoder()
for column in df_train.columns:
    if df_train[column].dtype == 'object':
        df_train[column] = label_encode.fit_transform(df_train[column])


In [10]:
df_train.drop(['Id','Street','YearRemodAdd','MiscVal'], axis=1, inplace= True)

In [11]:
df_train['MSSubClass'] = df_train['MSSubClass'].apply(str)
df_train['OverallCond'] = df_train['OverallCond'].astype(str)
df_train['YrSold'] = df_train['YrSold'].astype(str)
df_train['MoSold'] = df_train['MoSold'].astype(str)

In [12]:
print("--------------------Null values ratio-----------------------\n")
display(df_test.isnull().mean()*100)

--------------------Null values ratio-----------------------



Id                0.000000
MSSubClass        0.000000
MSZoning          0.274160
LotFrontage      15.558602
LotArea           0.000000
                   ...    
MiscVal           0.000000
MoSold            0.000000
YrSold            0.000000
SaleType          0.068540
SaleCondition     0.000000
Length: 80, dtype: float64

In [13]:
df_test.columns[df_test.isnull().mean()*100 > 20]

Index(['Alley', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature'], dtype='object')

In [14]:
df_test = df_test.drop(columns=['Alley','FireplaceQu','PoolQC','Fence','MiscFeature'],axis=1)

In [15]:
for column in df_test.columns:
    if df_test[column].dtype == np.object:
        df_test[column].fillna(df_test[column].mode()[0], inplace=True)
    elif df_test[column].dtype == np.float64:
        df_test[column].fillna(df_test[column].median(), inplace=True)
    elif df_test[column].dtype == np.int64:
        df_test[column].fillna(df_test[column].median(), inplace=True)
        

In [16]:
print("---------------Checking for the null value again-----------------\n")
display(df_test.isnull().sum())
print("The totall amount of null values:",df_test.isnull().sum().sum())
print("---------------------The final test data set-------------------\n")
display(df_test)

---------------Checking for the null value again-----------------



Id               0
MSSubClass       0
MSZoning         0
LotFrontage      0
LotArea          0
                ..
MiscVal          0
MoSold           0
YrSold           0
SaleType         0
SaleCondition    0
Length: 75, dtype: int64

The totall amount of null values: 0
---------------------The final test data set-------------------



,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,120,0,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,IR1,Lvl,AllPub,Corner,...,36,0,0,0,0,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,IR1,Lvl,AllPub,Inside,...,34,0,0,0,0,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,IR1,Lvl,AllPub,Inside,...,36,0,0,0,0,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,IR1,HLS,AllPub,Inside,...,82,0,0,144,0,0,1,2010,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,2915,160,RM,21.0,1936,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,0,0,0,6,2006,WD,Normal
1455,2916,160,RM,21.0,1894,Pave,Reg,Lvl,AllPub,Inside,...,24,0,0,0,0,0,4,2006,WD,Abnorml
1456,2917,20,RL,160.0,20000,Pave,Reg,Lvl,AllPub,Inside,...,0,0,0,0,0,0,9,2006,WD,Abnorml
1457,2918,85,RL,62.0,10441,Pave,Reg,Lvl,AllPub,Inside,...,32,0,0,0,0,700,7,2006,WD,Normal


In [17]:
df_test.drop(['PoolArea'], axis=1, inplace=True)

In [18]:
object_features = []
for column_name in df_test.columns:
    if df_test[column_name].dtype == 'object':
        object_features.append(column_name) 

print(f"----------------Object type features name ----------> number of object features:{len(object_features)} <----------\n ")        
display(object_features)

----------------Object type features name ----------> number of object features:38 <----------
 


['MSZoning',
 'Street',
 'LotShape',
 'LandContour',
 'Utilities',
 'LotConfig',
 'LandSlope',
 'Neighborhood',
 'Condition1',
 'Condition2',
 'BldgType',
 'HouseStyle',
 'RoofStyle',
 'RoofMatl',
 'Exterior1st',
 'Exterior2nd',
 'MasVnrType',
 'ExterQual',
 'ExterCond',
 'Foundation',
 'BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinType2',
 'Heating',
 'HeatingQC',
 'CentralAir',
 'Electrical',
 'KitchenQual',
 'Functional',
 'GarageType',
 'GarageFinish',
 'GarageQual',
 'GarageCond',
 'PavedDrive',
 'SaleType',
 'SaleCondition']

In [19]:
label_encode = LabelEncoder()
for column in df_test.columns:
    if df_test[column].dtype == 'object':
        df_test[column] = label_encode.fit_transform(df_test[column])

In [20]:
df_test.drop(['Street','YearRemodAdd','MiscVal'], axis=1, inplace= True)

In [21]:
df_test['MSSubClass'] = df_test['MSSubClass'].apply(str)
df_test['OverallCond'] = df_test['OverallCond'].astype(str)
df_test['YrSold'] = df_test['YrSold'].astype(str)
df_test['MoSold'] = df_test['MoSold'].astype(str)

In [22]:
X=df_train.drop(columns=['SalePrice'])
Y=df_train['SalePrice']
X.df_test=df_test

In [23]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,random_state=42,test_size=0.3)

In [24]:
X_train= np.asarray(X_train).astype(np.int)
Y_train= np.asarray(Y_train).astype(np.int)
X_test= np.asarray(X_test).astype(np.int)
Y_test= np.asarray(Y_test).astype(np.int)

**Build the Model**

In [25]:
tf.random.set_seed(42)
norm_layer = tf.keras.layers.Normalization(input_shape=X_train.shape[1:])
model = tf.keras.Sequential([
    norm_layer,
    tf.keras.layers.Dense(200, activation="relu"),
    tf.keras.layers.Dense(200, activation="relu"),
    tf.keras.layers.Dense(200, activation="relu"),
    tf.keras.layers.Dense(200, activation="relu"),
    tf.keras.layers.Dense(200, activation="relu"),
    tf.keras.layers.Dense(200, activation="relu"),
    tf.keras.layers.Dense(200, activation="relu"),
    tf.keras.layers.Dense(200, activation="relu"),
    tf.keras.layers.Dense(200, activation="relu"),
    
    tf.keras.layers.Dense(1)
])

**Compile the model**

In [26]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(loss="mse", optimizer=optimizer, metrics=["RootMeanSquaredError"])
norm_layer.adapt(X_train)

In [27]:
model.fit(X_train, Y_train, epochs=4500)

Epoch 1/4500
32/32 [==============================] - 2s 7ms/step - loss: 35240759296.0000 - root_mean_squared_error: 187725.2188
Epoch 2/4500
32/32 [==============================] - 0s 6ms/step - loss: 4557330944.0000 - root_mean_squared_error: 67508.0078
Epoch 3/4500
32/32 [==============================] - 0s 7ms/step - loss: 1760728448.0000 - root_mean_squared_error: 41961.0352
Epoch 4/4500
32/32 [==============================] - 0s 6ms/step - loss: 1574239616.0000 - root_mean_squared_error: 39676.6875
Epoch 5/4500
32/32 [==============================] - 0s 6ms/step - loss: 1150281216.0000 - root_mean_squared_error: 33915.7969
Epoch 6/4500
32/32 [==============================] - 0s 6ms/step - loss: 907740800.0000 - root_mean_squared_error: 30128.7363
Epoch 7/4500
32/32 [==============================] - 0s 6ms/step - loss: 888714560.0000 - root_mean_squared_error: 29811.3164
Epoch 8/4500
32/32 [==============================] - 0s 6ms/step - loss: 922868032.0000 - root_mean_squ

**Evaluate the Model**

In [28]:
mse_test, rmse_test = model.evaluate(X_test, Y_test)

14/14 [==============================] - 0s 3ms/step - loss: 982625984.0000 - root_mean_squared_error: 31346.8652


In [29]:
rmse_test

31346.865234375